In [1]:

import subprocess, sys
def pip_run(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + list(args)
    print("Running:", " ".join(cmd)); subprocess.check_call(cmd)

pip_run("--upgrade", "pip", "setuptools", "wheel", "setuptools_scm")
pip_run("seqeval", "evaluate")

import numpy, pandas, transformers, datasets, evaluate, seqeval, accelerate
for m in [numpy, pandas, transformers, datasets, evaluate, accelerate]:
    print(f"  OK  {m.__name__}=={getattr(m, '__version__', '?')}")
print("  OK  seqeval (installed)")



from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token, add_to_git_credential=False)
    print("Authenticated with Hugging Face (via Kaggle Secrets)")
except Exception:
    import os
    hf_token = os.environ.get("HF_TOKEN", None)
    if hf_token:
        login(token=hf_token, add_to_git_credential=False)
        print("Authenticated with Hugging Face (via env var)")
    else:
        print("!! No HF_TOKEN found -- proceeding unauthenticated.")
        print("   This worked for languages hi..as last run but is more likely")
        print("   to hit transient Hub errors over a long, many-language run.")
        print("   Strongly recommended: Kaggle -> Add-ons -> Secrets -> HF_TOKEN")


import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import time, shutil, random, json, itertools, gc
import numpy as np
import pandas as pd
import torch
from torch.nn import CrossEntropyLoss
import matplotlib.pyplot as plt
from datasets import load_dataset, DatasetDict, Dataset, Features, ClassLabel, Sequence, Value
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification, set_seed,
)
import evaluate

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

SMOKE_TEST = False

CONFIG = dict(
    languages   = ["hi", "bn", "mr", "gu", "pa", "or", "as", "ur",
                   "ta", "te", "kn", "ml"],
    source_lang = "en",
    models = {
        "mBERT":     "bert-base-multilingual-cased",
        "XLM-R":     "xlm-roberta-base",
        "MuRIL":     "google/muril-base-cased",
        "IndicBERT": "ai4bharat/IndicBERTv2-MLM-only",
    },
    seeds       = [42, 123, 2024],
    k_values    = [10, 50, 100],

    experiments = dict(zero_shot=True, few_shot=True, sequential=True),

    total_budget_hours  = 8.0,
    min_run_headroom_s  = 240,

    zs_train_cap  = 8000,
    zs_epochs     = 2,
    zs_batch      = 32,
    zs_lr         = 3e-5,

    fs_epochs_by_k = {10: 20, 50: 12, 100: 8},
    fs_batch_by_k  = {10: 4,  50: 6,  100: 8},
    fs_lr_by_model = {"mBERT": 2e-5, "XLM-R": 2e-5, "MuRIL": 1e-5, "IndicBERT": 1.5e-5},

    max_len       = 96,
    weight_decay  = 0.01,
    warmup_ratio  = 0.1,
    max_grad_norm = 1.0,
    use_weighted_loss     = True,
    collapse_f1_threshold = 0.01,
    max_collapse_retries  = 1,
    retry_lr_factor       = 0.5,
    val_cap_fewshot = 200,
    test_eval_cap   = 500,

    use_lalf = True,
    lalf_freeze_by_lang = {
        "hi": 4, "bn": 4, "mr": 4, "gu": 4, "pa": 4, "or": 4, "as": 4,
        "ur": 2,
        "ta": 4, "te": 4, "kn": 4, "ml": 4,
    },

    # --- Hub download resilience (new in v4.5) ---
    hub_max_retries = 5,
    hub_backoff_s   = 8,   # doubles each retry: 8, 16, 32, 64, 128
)

if SMOKE_TEST:
    CONFIG["languages"] = ["hi"]
    CONFIG["models"]    = {"mBERT": CONFIG["models"]["mBERT"]}
    CONFIG["seeds"]     = [42]
    CONFIG["k_values"]  = [10]
    print("!! SMOKE_TEST=True -- pipeline sanity check, NOT paper numbers.")
else:
    print("!! FULL RUN -- 12 languages x 4 models x 3 seeds x 3 k-values.")

OUTPUT_DIR = "/kaggle/working/results"
CKPT_DIR   = "/kaggle/working/_ckpts"
for d in (OUTPUT_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)

RESULTS_CSV = os.path.join(OUTPUT_DIR, "all_results_raw.csv")
FAILED_CSV  = os.path.join(OUTPUT_DIR, "failed_runs.csv")
with open(os.path.join(OUTPUT_DIR, "experiment_config.json"), "w") as f:
    json.dump({**CONFIG, "models": list(CONFIG["models"].keys())}, f, indent=2, default=str)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("Models:", list(CONFIG["models"].keys()))
print("Langs:", CONFIG["languages"])
print("Seeds:", CONFIG["seeds"], "| k:", CONFIG["k_values"])
print("Experiments:", CONFIG["experiments"])

TOTAL_BUDGET_SEC = CONFIG["total_budget_hours"] * 3600
START_TIME = time.time()
def time_left():     return TOTAL_BUDGET_SEC - (time.time() - START_TIME)
def time_used_str(): return f"{(time.time() - START_TIME)/3600:.2f}h"

metric = evaluate.load("seqeval")



def run_key(experiment, model, lang, k, seed):
    return f"{experiment}|{model}|{lang}|{k}|{seed}"

if os.path.exists(RESULTS_CSV):
    prev = pd.read_csv(RESULTS_CSV)
    completed_keys = set(
        run_key(r["experiment"], r["model"],
                r["lang"] if pd.notna(r["lang"]) else "",
                r["k"] if pd.notna(r["k"]) else "",
                r["seed"])
        for _, r in prev.iterrows()
    )
    all_results = prev.to_dict("records")
    print(f"!! Resume: {len(prev)} rows loaded -> {len(completed_keys)} unique runs")
else:
    completed_keys = set()
    all_results = []

error_examples = []
collapsed_runs = []
failed_runs = []   # NEW in v4.5: runs that errored out entirely (not just collapsed)

def save_progress():
    if all_results:
        pd.DataFrame(all_results).to_csv(RESULTS_CSV, index=False)

def save_failed():
    if failed_runs:
        pd.DataFrame(failed_runs).to_csv(FAILED_CSV, index=False)



def robust_load_dataset(*args, max_retries=None, backoff_s=None, **kwargs):
    """Wraps datasets.load_dataset with retries + exponential backoff.

    The Hub occasionally returns a transient error mid-run; the datasets
    library then tries to fall back to a local cache, and if that exact
    config was never cached, it raises instead of retrying. This wrapper
    retries the actual Hub call several times before giving up, which is
    what crashed Phase B on 'ur' last run.
    """
    max_retries = max_retries or CONFIG["hub_max_retries"]
    backoff_s = backoff_s or CONFIG["hub_backoff_s"]
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            return load_dataset(*args, **kwargs)
        except Exception as e:
            last_err = e
            if attempt < max_retries:
                wait = backoff_s * (2 ** (attempt - 1))
                print(f"   !! Hub load failed (attempt {attempt}/{max_retries}): "
                      f"{type(e).__name__}: {str(e)[:200]}")
                print(f"      retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"   !! Hub load failed after {max_retries} attempts, giving up.")
    raise last_err



CANDIDATE_LABEL_ORDER = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]

def get_label_names(dd_or_ds):
    ds = dd_or_ds[list(dd_or_ds.keys())[0]] if isinstance(dd_or_ds, DatasetDict) else dd_or_ds
    return list(ds.features["ner_tags"].feature.names)

def remap_tags_to_canonical(dd_or_ds, canonical):
    names = get_label_names(dd_or_ds)
    if names == canonical:
        return dd_or_ds, canonical
    idx = {n: i for i, n in enumerate(canonical)}
    def fix(ex):
        return {"tokens": ex["tokens"],
                "ner_tags": [idx.get(names[i], 0) for i in ex["ner_tags"]]}
    feats = Features({"tokens": Sequence(Value("string")),
                      "ner_tags": Sequence(ClassLabel(names=canonical))})
    if isinstance(dd_or_ds, DatasetDict):
        for s in dd_or_ds:
            dd_or_ds[s] = dd_or_ds[s].map(fix, features=feats)
    else:
        dd_or_ds = dd_or_ds.map(fix, features=feats)
    return dd_or_ds, canonical

canonical_labels = CANDIDATE_LABEL_ORDER
LABEL_LIST = canonical_labels
LABEL2ID   = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL   = {i: l for i, l in enumerate(LABEL_LIST)}
NUM_LABELS = len(LABEL_LIST)


def load_wikiann(lang, canonical):
    dd = robust_load_dataset("unimelb-nlp/wikiann", lang)
    dd, _ = remap_tags_to_canonical(dd, canonical)
    return dd

wikiann_source = load_wikiann(CONFIG["source_lang"], canonical_labels)
if CONFIG["zs_train_cap"] and len(wikiann_source["train"]) > CONFIG["zs_train_cap"]:
    wikiann_source["train"] = (
        wikiann_source["train"].shuffle(seed=42).select(range(CONFIG["zs_train_cap"]))
    )

wikiann_targets = {l: load_wikiann(l, canonical_labels) for l in CONFIG["languages"]}

print("WikiAnn EN train:", len(wikiann_source["train"]))
for l in CONFIG["languages"]:
    print(f"WikiAnn {l} test:", len(wikiann_targets[l]["test"]))


def load_naamapadam(lang, canonical, k_shot=None, seed=42, eval_cap=500, val_cap=200):
    dd = robust_load_dataset("ai4bharat/naamapadam", data_dir=lang,
                             revision="refs/convert/parquet", trust_remote_code=False)
    if "validation" not in dd:
        split = dd["train"].train_test_split(test_size=0.1, seed=seed)
        dd["train"], dd["validation"] = split["train"], split["test"]
    dd, _ = remap_tags_to_canonical(dd, canonical)
    if k_shot is not None:
        n = min(k_shot, len(dd["train"]))
        dd["train"] = dd["train"].shuffle(seed=seed).select(range(n))
    if val_cap and len(dd["validation"]) > val_cap:
        dd["validation"] = dd["validation"].shuffle(seed=seed).select(range(val_cap))
    if eval_cap and len(dd["test"]) > eval_cap:
        dd["test"] = dd["test"].shuffle(seed=seed).select(range(eval_cap))
    return dd


def make_tokenize_fn(tokenizer):
    def tok_fn(examples):
        tok = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True,
                        max_length=CONFIG["max_len"], padding=False)
        labels_all = []
        for i, labs in enumerate(examples["ner_tags"]):
            word_ids = tok.word_ids(batch_index=i)
            prev, seq = None, []
            for wid in word_ids:
                if wid is None:    seq.append(-100)
                elif wid != prev:  seq.append(labs[wid])
                else:              seq.append(-100)
                prev = wid
            labels_all.append(seq)
        tok["labels"] = labels_all
        return tok
    return tok_fn


def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=2)
    tp, tl = [], []
    for p, l in zip(preds, labels):
        ps = [ID2LABEL[pi] for pi, li in zip(p, l) if li != -100]
        ls = [ID2LABEL[li] for li in l if li != -100]
        tp.append(ps); tl.append(ls)
    res = metric.compute(predictions=tp, references=tl, zero_division=0)
    out = {"precision": res["overall_precision"], "recall": res["overall_recall"],
           "f1": res["overall_f1"], "accuracy": res["overall_accuracy"]}
    for ent in ["PER", "ORG", "LOC"]:
        if ent in res:
            out[f"f1_{ent}"] = res[ent]["f1"]
            out[f"p_{ent}"]  = res[ent]["precision"]
            out[f"r_{ent}"]  = res[ent]["recall"]
        else:
            out[f"f1_{ent}"] = np.nan; out[f"p_{ent}"] = np.nan; out[f"r_{ent}"] = np.nan
    return out


def compute_class_weights(train_ds, num_labels):
    counts = np.ones(num_labels)
    for ex in train_ds:
        for l in ex["ner_tags"]:
            counts[l] += 1
    inv = 1.0 / counts
    return torch.tensor(inv / inv.sum() * num_labels, dtype=torch.float)


class WeightedTrainer(Trainer):
    """Weighted CE to prevent all-O collapse; DataParallel-safe."""
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        m = self.model
        while hasattr(m, "module"): m = m.module
        self._num_labels = int(m.config.num_labels)

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        w = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = CrossEntropyLoss(weight=w, ignore_index=-100)(
            logits.view(-1, self._num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def apply_lalf(model, lang, use_lalf):
    """Language-Adaptive Layer Freezing: freeze bottom-N encoder layers."""
    if not use_lalf:
        return 0
    n = CONFIG["lalf_freeze_by_lang"].get(lang, 0)
    if n <= 0:
        return 0
    enc = None
    for attr in ("bert", "roberta"):
        if hasattr(model, attr):
            base = getattr(model, attr)
            if hasattr(base, "encoder") and hasattr(base.encoder, "layer"):
                enc = base.encoder.layer
                break
    if enc is None:
        return 0
    n = min(n, len(enc))
    for i in range(n):
        for p in enc[i].parameters():
            p.requires_grad = False
    return n


def build_and_train(model_name, ckpt, train_ds, run_name, epochs, batch, seed, lr,
                    use_weights, lang="", init_from_ckpt=None, use_lalf=False):
    set_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(ckpt)
    base = init_from_ckpt if init_from_ckpt else ckpt
    model = AutoModelForTokenClassification.from_pretrained(
        base, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID
    ).to(DEVICE)

    frozen = apply_lalf(model, lang, use_lalf)
    if frozen:
        print(f"   LALF: froze bottom {frozen} layers for lang={lang}")

    tok_fn = make_tokenize_fn(tokenizer)
    train_tok = train_ds.map(tok_fn, batched=True, remove_columns=train_ds.column_names)
    collator = DataCollatorForTokenClassification(tokenizer)

    steps_per_epoch = max(1, len(train_tok) // max(1, batch))
    warmup_steps = max(3, int(CONFIG["warmup_ratio"] * steps_per_epoch * epochs))

    args = TrainingArguments(
        output_dir=os.path.join(CKPT_DIR, run_name),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch,
        per_device_eval_batch_size=32,
        learning_rate=lr,
        weight_decay=CONFIG["weight_decay"],
        warmup_steps=warmup_steps,
        max_grad_norm=CONFIG["max_grad_norm"],
        logging_steps=max(20, steps_per_epoch),
        eval_strategy="no",
        save_strategy="no",
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=seed, data_seed=seed,
        disable_tqdm=True,
    )
    kwargs = dict(model=model, args=args, train_dataset=train_tok,
                  data_collator=collator, compute_metrics=compute_metrics)
    if use_weights:
        kwargs["class_weights"] = compute_class_weights(train_ds, NUM_LABELS)
    trainer = (WeightedTrainer if use_weights else Trainer)(**kwargs)
    trainer.train()
    return trainer, tokenizer


def quick_val_f1(trainer, tokenizer, val_ds):
    tok_fn = make_tokenize_fn(tokenizer)
    vt = val_ds.map(tok_fn, batched=True, remove_columns=val_ds.column_names)
    preds = trainer.predict(vt)
    return compute_metrics((preds.predictions, preds.label_ids))["f1"]


def train_with_guard(model_name, ckpt, train_ds, val_ds, run_name, epochs, batch, seed,
                     lr, use_weights, lang="", init_from_ckpt=None, use_lalf=False):
    """Train, retry once with halved LR if val F1 collapses."""
    attempt, current_lr = 0, lr
    while True:
        trainer, tokenizer = build_and_train(
            model_name, ckpt, train_ds, run_name, epochs, batch, seed,
            current_lr, use_weights, lang=lang,
            init_from_ckpt=init_from_ckpt, use_lalf=use_lalf,
        )
        v_f1 = quick_val_f1(trainer, tokenizer, val_ds)
        collapsed = v_f1 < CONFIG["collapse_f1_threshold"]
        if collapsed and attempt < CONFIG["max_collapse_retries"]:
            attempt += 1
            current_lr *= CONFIG["retry_lr_factor"]
            print(f"   !! val collapse (F1={v_f1:.4f}); retry {attempt} lr={current_lr:.2e}")
            shutil.rmtree(os.path.join(CKPT_DIR, run_name), ignore_errors=True)
            del trainer; gc.collect(); torch.cuda.empty_cache()
            continue
        return trainer, tokenizer, v_f1, attempt


def eval_on(trainer, tokenizer, eval_ds, meta, save_errors=False, tag=""):
    tok_fn = make_tokenize_fn(tokenizer)
    etok = eval_ds.map(tok_fn, batched=True, remove_columns=eval_ds.column_names)
    preds = trainer.predict(etok)
    m = compute_metrics((preds.predictions, preds.label_ids))
    m.update(meta)
    all_results.append(m)
    save_progress()

    collapsed = ((m["f1"] < CONFIG["collapse_f1_threshold"]) and
                 (m["recall"] < CONFIG["collapse_f1_threshold"]))
    flag = "  !! COLLAPSE" if collapsed else ""
    print(f"   -> {meta.get('run')} | eval={meta.get('eval_on')} | "
          f"F1={m['f1']:.4f} (P={m['precision']:.4f} R={m['recall']:.4f}) | "
          f"{time_used_str()}{flag}")
    if collapsed:
        collapsed_runs.append({**meta, "f1": m["f1"]})

    if save_errors and len(error_examples) < 200:
        pid = np.argmax(preds.predictions, axis=2)
        for i, (p_row, l_row) in enumerate(zip(pid, preds.label_ids)):
            ps = [ID2LABEL[p] for p, l in zip(p_row, l_row) if l != -100]
            ls = [ID2LABEL[l] for l in l_row if l != -100]
            if ps != ls:
                error_examples.append({
                    "tag": tag, "tokens": list(eval_ds[i]["tokens"])[:80],
                    "gold": ls[:80], "pred": ps[:80],
                })
                if len(error_examples) >= 200:
                    break



zs_ckpts = {}

if CONFIG["experiments"]["zero_shot"]:
    print("=" * 70)
    print("PHASE A: ZERO-SHOT (EN -> target)")
    print("=" * 70)

    for model_name, ckpt in CONFIG["models"].items():
        for seed in CONFIG["seeds"]:
            run_name = f"zs_{model_name}_s{seed}"
            key = run_key("zero_shot", model_name, "", "", seed)
            ckpt_path = os.path.join(CKPT_DIR, run_name)

            if key in completed_keys and os.path.exists(ckpt_path):
                print(f"\n--- {run_name} : already completed, reusing ckpt ---")
                zs_ckpts[(model_name, seed)] = ckpt_path
                continue

            if time_left() < 900:
                print(f"!! Skipping rest of Phase A ({time_used_str()})")
                break

            print(f"\n--- {run_name} ---")
            try:
                trainer, tok = build_and_train(
                    model_name, ckpt, wikiann_source["train"], run_name,
                    CONFIG["zs_epochs"], CONFIG["zs_batch"], seed, CONFIG["zs_lr"],
                    use_weights=False, use_lalf=False,
                )
                trainer.save_model(ckpt_path); tok.save_pretrained(ckpt_path)
                zs_ckpts[(model_name, seed)] = ckpt_path

                for lang in CONFIG["languages"]:
                    eval_on(
                        trainer, tok, wikiann_targets[lang]["test"],
                        meta=dict(experiment="zero_shot", model=model_name, eval_on=lang,
                                  lang=None, k=None, seed=seed, run=run_name),
                        save_errors=(seed == CONFIG["seeds"][0]),
                        tag=f"zs-{model_name}-{lang}",
                    )
                del trainer; gc.collect(); torch.cuda.empty_cache()
            except Exception as e:
                print(f"   !! {run_name} FAILED entirely: {type(e).__name__}: {str(e)[:300]}")
                failed_runs.append(dict(experiment="zero_shot", model=model_name,
                                        lang="", k="", seed=seed, run=run_name,
                                        error=f"{type(e).__name__}: {str(e)[:300]}"))
                save_failed()
                gc.collect(); torch.cuda.empty_cache()
                continue



if CONFIG["experiments"]["few_shot"]:
    print("=" * 70)
    print("PHASE B: FEW-SHOT (target-only, LALF on)")
    print("=" * 70)

    for model_name, ckpt in CONFIG["models"].items():
        for lang in CONFIG["languages"]:
            for k in CONFIG["k_values"]:
                for seed in CONFIG["seeds"]:
                    key = run_key("few_shot", model_name, lang, k, seed)
                    if key in completed_keys:
                        continue
                    if time_left() < CONFIG["min_run_headroom_s"]:
                        print(f"!! Skipping rest of Phase B ({time_used_str()})")
                        break

                    run_name = f"fs_{model_name}_{lang}_k{k}_s{seed}"
                    print(f"\n--- {run_name} ---")
                    try:
                        dd = load_naamapadam(lang, LABEL_LIST, k_shot=k, seed=seed,
                                             eval_cap=CONFIG["test_eval_cap"],
                                             val_cap=CONFIG["val_cap_fewshot"])
                        epochs = CONFIG["fs_epochs_by_k"][k]
                        batch  = min(CONFIG["fs_batch_by_k"][k], max(1, k))
                        lr     = CONFIG["fs_lr_by_model"].get(model_name, 2e-5)

                        trainer, tok, v_f1, retries = train_with_guard(
                            model_name, ckpt, dd["train"], dd["validation"], run_name,
                            epochs, batch, seed, lr, CONFIG["use_weighted_loss"],
                            lang=lang, use_lalf=CONFIG["use_lalf"],
                        )
                        eval_on(
                            trainer, tok, dd["test"],
                            meta=dict(experiment="few_shot", model=model_name, eval_on=lang,
                                      lang=lang, k=k, seed=seed, run=run_name,
                                      retries=retries),
                        )
                        shutil.rmtree(os.path.join(CKPT_DIR, run_name), ignore_errors=True)
                        del trainer; gc.collect(); torch.cuda.empty_cache()
                    except Exception as e:
                        print(f"   !! {run_name} FAILED entirely: {type(e).__name__}: {str(e)[:300]}")
                        failed_runs.append(dict(experiment="few_shot", model=model_name,
                                                lang=lang, k=k, seed=seed, run=run_name,
                                                error=f"{type(e).__name__}: {str(e)[:300]}"))
                        save_failed()
                        shutil.rmtree(os.path.join(CKPT_DIR, run_name), ignore_errors=True)
                        gc.collect(); torch.cuda.empty_cache()
                        continue



if CONFIG["experiments"]["sequential"]:
    print("=" * 70)
    print("PHASE C: SEQUENTIAL (EN-zero-shot -> target few-shot, LALF on)")
    print("=" * 70)

    for model_name in CONFIG["models"]:
        for seed in CONFIG["seeds"]:
            base = zs_ckpts.get((model_name, seed))
            if base is None:
                print(f"  [skip] no Phase A ckpt for {model_name} seed={seed}")
                continue
            for lang in CONFIG["languages"]:
                for k in CONFIG["k_values"]:
                    key = run_key("sequential", model_name, lang, k, seed)
                    if key in completed_keys:
                        continue
                    if time_left() < CONFIG["min_run_headroom_s"]:
                        print(f"!! Skipping rest of Phase C ({time_used_str()})")
                        break

                    run_name = f"seq_{model_name}_{lang}_k{k}_s{seed}"
                    print(f"\n--- {run_name} ---")
                    try:
                        dd = load_naamapadam(lang, LABEL_LIST, k_shot=k, seed=seed,
                                             eval_cap=CONFIG["test_eval_cap"],
                                             val_cap=CONFIG["val_cap_fewshot"])
                        epochs = CONFIG["fs_epochs_by_k"][k]
                        batch  = min(CONFIG["fs_batch_by_k"][k], max(1, k))
                        lr     = CONFIG["fs_lr_by_model"].get(model_name, 2e-5)

                        trainer, tok, v_f1, retries = train_with_guard(
                            model_name, base, dd["train"], dd["validation"], run_name,
                            epochs, batch, seed, lr, CONFIG["use_weighted_loss"],
                            lang=lang, init_from_ckpt=base, use_lalf=CONFIG["use_lalf"],
                        )
                        eval_on(
                            trainer, tok, dd["test"],
                            meta=dict(experiment="sequential", model=model_name, eval_on=lang,
                                      lang=lang, k=k, seed=seed, run=run_name,
                                      retries=retries),
                        )
                        shutil.rmtree(os.path.join(CKPT_DIR, run_name), ignore_errors=True)
                        del trainer; gc.collect(); torch.cuda.empty_cache()
                    except Exception as e:
                        print(f"   !! {run_name} FAILED entirely: {type(e).__name__}: {str(e)[:300]}")
                        failed_runs.append(dict(experiment="sequential", model=model_name,
                                                lang=lang, k=k, seed=seed, run=run_name,
                                                error=f"{type(e).__name__}: {str(e)[:300]}"))
                        save_failed()
                        shutil.rmtree(os.path.join(CKPT_DIR, run_name), ignore_errors=True)
                        gc.collect(); torch.cuda.empty_cache()
                        continue



results_df = pd.DataFrame(all_results)
results_df.to_csv(RESULTS_CSV, index=False)

if failed_runs:
    print(f"\n!! {len(failed_runs)} runs failed entirely (see failed_runs.csv). "
          f"Re-run this notebook to retry them -- completed runs are skipped automatically.")

if collapsed_runs:
    coll_df = pd.DataFrame(collapsed_runs)
    coll_df.to_csv(os.path.join(OUTPUT_DIR, "collapsed_runs.csv"), index=False)
    print(f"\n!! {len(collapsed_runs)} runs collapsed after retries. See collapsed_runs.csv")
    print(coll_df[["run", "model", "lang", "k", "f1"]].to_string(index=False))
else:
    print("\nNo collapsed runs.")

agg_cols = ["precision", "recall", "f1", "accuracy", "f1_PER", "f1_ORG", "f1_LOC"]
group_keys = ["experiment", "model", "eval_on", "lang", "k"]

mean_df = results_df.groupby(group_keys, dropna=False)[agg_cols].mean().reset_index()
std_df  = results_df.groupby(group_keys, dropna=False)[agg_cols].std().reset_index()
std_df.columns = group_keys + [c + "_std" for c in agg_cols]
summary_df = mean_df.merge(std_df, on=group_keys, how="left")
summary_df.to_csv(os.path.join(OUTPUT_DIR, "summary_mean_std.csv"), index=False)
print("\nSummary (head 20):")
print(summary_df.head(20).to_string())



def paired_bootstrap(a, b, n=2000, seed=42):
    rng = np.random.default_rng(seed)
    a, b = np.asarray(a), np.asarray(b)
    diffs = np.empty(n)
    for i in range(n):
        idx = rng.integers(0, len(a), len(a))
        diffs[i] = (a[idx] - b[idx]).mean()
    return float((diffs <= 0).mean())

sig_rows = []
for exp in results_df["experiment"].unique():
    sub = results_df[results_df["experiment"] == exp]
    for (lang, k), g in sub.groupby(["lang", "k"], dropna=False):
        pivot = g.pivot_table(index="seed", columns="model", values="f1")
        if len(pivot) < 2:
            continue
        for m1, m2 in itertools.combinations(pivot.columns.tolist(), 2):
            if pivot[m1].notna().all() and pivot[m2].notna().all():
                p = paired_bootstrap(pivot[m1].values, pivot[m2].values)
                sig_rows.append({
                    "experiment": exp, "lang": lang, "k": k,
                    "model_a": m1, "model_b": m2,
                    "mean_a": pivot[m1].mean(), "mean_b": pivot[m2].mean(),
                    "p_a_gt_b": p, "n_seeds": len(pivot),
                })
sig_df = pd.DataFrame(sig_rows)
sig_df.to_csv(os.path.join(OUTPUT_DIR, "significance.csv"), index=False)
if not sig_df.empty:
    print(sig_df.to_string())



def save_bar(pivot, title, fname, ylabel="F1"):
    if pivot.empty: return
    ax = pivot.plot(kind="bar", figsize=(max(7, 1.2 * len(pivot)), 5))
    ax.set_ylabel(ylabel); ax.set_title(title); ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, fname), dpi=150); plt.close()

zs = summary_df[summary_df["experiment"] == "zero_shot"]
if not zs.empty:
    save_bar(zs.pivot_table(index="eval_on", columns="model", values="f1"),
             "Zero-Shot Cross-Lingual NER F1 (EN WikiAnn)", "plot_zeroshot.png")

fs = summary_df[summary_df["experiment"] == "few_shot"]
if not fs.empty:
    for lang in fs["lang"].dropna().unique():
        sub = fs[fs["lang"] == lang].pivot_table(index="k", columns="model", values="f1")
        if not sub.empty:
            ax = sub.plot(marker="o", figsize=(7, 5))
            ax.set_xlabel("k"); ax.set_ylabel("F1")
            ax.set_title(f"Few-shot F1 vs k — {lang.upper()}"); ax.set_ylim(0, 1)
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_DIR, f"plot_fewshot_{lang}.png"), dpi=150)
            plt.close()

seq = summary_df[summary_df["experiment"] == "sequential"]
if not seq.empty:
    for lang in seq["lang"].dropna().unique():
        fs_l, seq_l = fs[fs["lang"] == lang], seq[seq["lang"] == lang]
        if fs_l.empty or seq_l.empty: continue
        fig, ax = plt.subplots(figsize=(8, 5))
        for model in fs_l["model"].unique():
            f_sub = fs_l[fs_l["model"] == model].sort_values("k")
            s_sub = seq_l[seq_l["model"] == model].sort_values("k")
            ax.plot(f_sub["k"], f_sub["f1"], "o--", label=f"{model} (few-shot)")
            ax.plot(s_sub["k"], s_sub["f1"], "s-",  label=f"{model} (seq)")
        ax.set_xlabel("k"); ax.set_ylabel("F1"); ax.set_ylim(0, 1)
        ax.set_title(f"Sequential vs few-shot — {lang.upper()}")
        ax.legend(fontsize=8); plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, f"plot_seq_vs_fs_{lang}.png"), dpi=150)
        plt.close()

if not zs.empty:
    for lang in zs["eval_on"].unique():
        pivot = zs[zs["eval_on"] == lang].set_index("model")[["f1_PER", "f1_ORG", "f1_LOC"]]
        save_bar(pivot, f"Per-entity F1 — zero-shot {lang.upper()}",
                 f"plot_perent_zs_{lang}.png")



if error_examples:
    err_df = pd.DataFrame(error_examples)
    err_df.to_csv(os.path.join(OUTPUT_DIR, "error_examples.csv"), index=False)
    for tag, g in err_df.groupby("tag"):
        print(f"\n=== {tag} (showing 2 of {len(g)}) ===")
        for _, row in g.head(2).iterrows():
            print("TOKENS:", " ".join(row["tokens"][:25]))
            print("GOLD  :", " ".join(row["gold"][:25]))
            print("PRED  :", " ".join(row["pred"][:25]))
            print("-")



print(f"\nTotal run time: {time_used_str()}")
print("Artifacts in:", OUTPUT_DIR)
for f in sorted(os.listdir(OUTPUT_DIR)):
    print("  ", f)

zip_base = "/kaggle/working/NER_Indian_Languages_Q1"
shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print(f"\nZipped: {zip_base}.zip")

Running: /usr/bin/python3 -m pip install -q --upgrade pip setuptools wheel setuptools_scm
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.2/122.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 6.3 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


Running: /usr/bin/python3 -m pip install -q seqeval evaluate
  OK  numpy==2.0.2
  OK  pandas==2.3.3
  OK  transformers==5.0.0
  OK  datasets==5.0.0
  OK  evaluate==0.4.6
  OK  accelerate==1.13.0
  OK  seqeval (installed)
!! No HF_TOKEN found -- proceeding unauthenticated.
   This worked for languages hi..as last run but is more likely
   to hit transient Hub errors over a long, many-language run.
   Strongly recommended: Kaggle -> Add-ons -> Secrets -> HF_TOKEN
!! FULL RUN -- 12 languages x 4 models x 3 seeds x 3 k-values.
Device: cuda
Models: ['mBERT', 'XLM-R', 'MuRIL', 'IndicBERT']
Langs: ['hi', 'bn', 'mr', 'gu', 'pa', 'or', 'as', 'ur', 'ta', 'te', 'kn', 'ml']
Seeds: [42, 123, 2024] | k: [10, 50, 100]
Experiments: {'zero_shot': True, 'few_shot': True, 'sequential': True}


README.md: 0.00B [00:00, ?B/s]

en/validation-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/test-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

... (5,505 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   LALF: froze bottom 4 layers for lang=ml


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

{'loss': '1.177', 'grad_norm': '6.599', 'learning_rate': '1.342e-05', 'epoch': '1.538'}
{'loss': '0.7731', 'grad_norm': '6.685', 'learning_rate': '1.026e-05', 'epoch': '3.077'}
{'loss': '0.5903', 'grad_norm': '7.654', 'learning_rate': '7.105e-06', 'epoch': '4.615'}
{'loss': '0.4293', 'grad_norm': '2.795', 'learning_rate': '3.947e-06', 'epoch': '6.154'}
{'loss': '0.4675', 'grad_norm': '2.98', 'learning_rate': '7.895e-07', 'epoch': '7.692'}
{'train_runtime': '10.83', 'train_samples_per_second': '73.89', 'train_steps_per_second': '9.606', 'train_loss': '0.6745', 'epoch': '8'}
   -> seq_IndicBERT_ml_k100_s2024 | eval=ml | F1=0.6631 (P=0.5728 R=0.7873) | 7.00h

!! 72 runs failed entirely (see failed_runs.csv). Re-run this notebook to retry them -- completed runs are skipped automatically.

!! 16 runs collapsed after retries. See collapsed_runs.csv
                      run     model lang     k       f1
             zs_mBERT_s42     mBERT None   NaN 0.000000
            zs_mBERT_s123     mBE

In [ ]:
# ================================================================================
# Cross-Lingual / Few-Shot NER for Low-Resource Indian Languages  (v4.5)
# Models : mBERT, XLM-RoBERTa, MuRIL, IndicBERTv2
# Data   : WikiAnn (EN->target zero-shot) + Naamapadam (target few-shot)
# Phases : A) zero-shot EN->target   B) direct few-shot   C) sequential
# Method : Language-Adaptive Layer Freezing (LALF)
# Langs  : all 12 Naamapadam Indian languages
#
# CHANGES FROM v4.4 (fixes the "ur" crash you saw):
#  1. robust_load_dataset(): wraps every load_dataset call with retries +
#     exponential backoff. The HF Hub occasionally returns a transient error,
#     datasets then silently falls back to a local cache -- and if that
#     language was never cached yet, it crashes instead of retrying the Hub.
#  2. Phase B and Phase C now wrap each run in try/except. A single flaky
#     download or training run is logged to failed_runs.csv and the loop
#     continues, instead of killing the remaining hours of the 12-language run.
#  3. Strong recommendation (see Cell 2): add HF_TOKEN via Kaggle Secrets.
#     Unauthenticated requests hit lower Hub rate limits and are more likely
#     to see exactly this kind of transient failure over a long multi-hour run.
# ================================================================================

# ============================= CELL 1: Setup =====================================
import subprocess, sys
def pip_run(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + list(args)
    print("Running:", " ".join(cmd)); subprocess.check_call(cmd)

pip_run("--upgrade", "pip", "setuptools", "wheel", "setuptools_scm")
pip_run("seqeval", "evaluate")

import numpy, pandas, transformers, datasets, evaluate, seqeval, accelerate
for m in [numpy, pandas, transformers, datasets, evaluate, accelerate]:
    print(f"  OK  {m.__name__}=={getattr(m, '__version__', '?')}")
print("  OK  seqeval (installed)")


# ============================= CELL 2: HF authentication =========================
# STRONGLY RECOMMENDED for this run: 12 languages x many downloads over several
# hours means many Hub requests. Unauthenticated requests are rate-limited
# harder and are more likely to hit the transient error that crashed Phase B
# on "ur" last time. Get a free token at https://huggingface.co/settings/tokens
# (read access is enough) and add it as a Kaggle Secret named HF_TOKEN.
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token, add_to_git_credential=False)
    print("Authenticated with Hugging Face (via Kaggle Secrets)")
except Exception:
    import os
    hf_token = os.environ.get("HF_TOKEN", None)
    if hf_token:
        login(token=hf_token, add_to_git_credential=False)
        print("Authenticated with Hugging Face (via env var)")
    else:
        print("!! No HF_TOKEN found -- proceeding unauthenticated.")
        print("   This worked for languages hi..as last run but is more likely")
        print("   to hit transient Hub errors over a long, many-language run.")
        print("   Strongly recommended: Kaggle -> Add-ons -> Secrets -> HF_TOKEN")


# ============================= CELL 3: Config ====================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import time, shutil, random, json, itertools, gc
import numpy as np
import pandas as pd
import torch
from torch.nn import CrossEntropyLoss
import matplotlib.pyplot as plt
from datasets import load_dataset, DatasetDict, Dataset, Features, ClassLabel, Sequence, Value
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification, set_seed,
)
import evaluate

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

SMOKE_TEST = False

CONFIG = dict(
    languages   = ["hi", "bn", "mr", "gu", "pa", "or", "as", "ur",
                   "ta", "te", "kn", "ml"],
    source_lang = "en",
    models = {
        "mBERT":     "bert-base-multilingual-cased",
        "XLM-R":     "xlm-roberta-base",
        "MuRIL":     "google/muril-base-cased",
        "IndicBERT": "ai4bharat/IndicBERTv2-MLM-only",
    },
    seeds       = [42, 123, 2024],
    k_values    = [10, 50, 100],

    experiments = dict(zero_shot=True, few_shot=True, sequential=True),

    total_budget_hours  = 8.0,
    min_run_headroom_s  = 240,

    zs_train_cap  = 8000,
    zs_epochs     = 2,
    zs_batch      = 32,
    zs_lr         = 3e-5,

    fs_epochs_by_k = {10: 20, 50: 12, 100: 8},
    fs_batch_by_k  = {10: 4,  50: 6,  100: 8},
    fs_lr_by_model = {"mBERT": 2e-5, "XLM-R": 2e-5, "MuRIL": 1e-5, "IndicBERT": 1.5e-5},

    max_len       = 96,
    weight_decay  = 0.01,
    warmup_ratio  = 0.1,
    max_grad_norm = 1.0,
    use_weighted_loss     = True,
    collapse_f1_threshold = 0.01,
    max_collapse_retries  = 1,
    retry_lr_factor       = 0.5,
    val_cap_fewshot = 200,
    test_eval_cap   = 500,

    use_lalf = True,
    lalf_freeze_by_lang = {
        "hi": 4, "bn": 4, "mr": 4, "gu": 4, "pa": 4, "or": 4, "as": 4,
        "ur": 2,
        "ta": 4, "te": 4, "kn": 4, "ml": 4,
    },

    # --- Hub download resilience (new in v4.5) ---
    hub_max_retries = 5,
    hub_backoff_s   = 8,   # doubles each retry: 8, 16, 32, 64, 128
)

if SMOKE_TEST:
    CONFIG["languages"] = ["hi"]
    CONFIG["models"]    = {"mBERT": CONFIG["models"]["mBERT"]}
    CONFIG["seeds"]     = [42]
    CONFIG["k_values"]  = [10]
    print("!! SMOKE_TEST=True -- pipeline sanity check, NOT paper numbers.")
else:
    print("!! FULL RUN -- 12 languages x 4 models x 3 seeds x 3 k-values.")

OUTPUT_DIR = "/kaggle/working/results"
CKPT_DIR   = "/kaggle/working/_ckpts"
for d in (OUTPUT_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)

RESULTS_CSV = os.path.join(OUTPUT_DIR, "all_results_raw.csv")
FAILED_CSV  = os.path.join(OUTPUT_DIR, "failed_runs.csv")
with open(os.path.join(OUTPUT_DIR, "experiment_config.json"), "w") as f:
    json.dump({**CONFIG, "models": list(CONFIG["models"].keys())}, f, indent=2, default=str)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("Models:", list(CONFIG["models"].keys()))
print("Langs:", CONFIG["languages"])
print("Seeds:", CONFIG["seeds"], "| k:", CONFIG["k_values"])
print("Experiments:", CONFIG["experiments"])

TOTAL_BUDGET_SEC = CONFIG["total_budget_hours"] * 3600
START_TIME = time.time()
def time_left():     return TOTAL_BUDGET_SEC - (time.time() - START_TIME)
def time_used_str(): return f"{(time.time() - START_TIME)/3600:.2f}h"

metric = evaluate.load("seqeval")


# ============================= CELL 4: Resume support =============================
def run_key(experiment, model, lang, k, seed):
    return f"{experiment}|{model}|{lang}|{k}|{seed}"

if os.path.exists(RESULTS_CSV):
    prev = pd.read_csv(RESULTS_CSV)
    completed_keys = set(
        run_key(r["experiment"], r["model"],
                r["lang"] if pd.notna(r["lang"]) else "",
                r["k"] if pd.notna(r["k"]) else "",
                r["seed"])
        for _, r in prev.iterrows()
    )
    all_results = prev.to_dict("records")
    print(f"!! Resume: {len(prev)} rows loaded -> {len(completed_keys)} unique runs")
else:
    completed_keys = set()
    all_results = []

error_examples = []
collapsed_runs = []
failed_runs = []   # NEW in v4.5: runs that errored out entirely (not just collapsed)

def save_progress():
    if all_results:
        pd.DataFrame(all_results).to_csv(RESULTS_CSV, index=False)

def save_failed():
    if failed_runs:
        pd.DataFrame(failed_runs).to_csv(FAILED_CSV, index=False)


# ============================= CELL 5: Robust Hub loading (NEW) ==================
def robust_load_dataset(*args, max_retries=None, backoff_s=None, **kwargs):
    """Wraps datasets.load_dataset with retries + exponential backoff.

    v4.5 found that a transient Hub error can leave a CORRUPTED/incomplete
    local cache entry behind (e.g. 'ur' last run). Plain retries then keep
    hitting that same broken cache and fail every time -- this is why all
    72 'ur' runs failed identically even with 5 retries each. Fix: after
    the first failure, escalate to download_mode='force_redownload' so a
    bad cache entry can't keep blocking every subsequent attempt.
    """
    max_retries = max_retries or CONFIG["hub_max_retries"]
    backoff_s = backoff_s or CONFIG["hub_backoff_s"]
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            call_kwargs = dict(kwargs)
            if attempt >= 2:
                # Escalate: bypass whatever is cached, in case it's corrupted.
                call_kwargs["download_mode"] = "force_redownload"
            return load_dataset(*args, **call_kwargs)
        except Exception as e:
            last_err = e
            if attempt < max_retries:
                wait = backoff_s * (2 ** (attempt - 1))
                print(f"   !! Hub load failed (attempt {attempt}/{max_retries}): "
                      f"{type(e).__name__}: {str(e)[:200]}")
                print(f"      retrying in {wait}s (force_redownload from attempt 2)...")
                time.sleep(wait)
            else:
                print(f"   !! Hub load failed after {max_retries} attempts, giving up.")
    raise last_err


# ============================= CELL 6: Label handling =============================
CANDIDATE_LABEL_ORDER = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]

def get_label_names(dd_or_ds):
    ds = dd_or_ds[list(dd_or_ds.keys())[0]] if isinstance(dd_or_ds, DatasetDict) else dd_or_ds
    return list(ds.features["ner_tags"].feature.names)

def remap_tags_to_canonical(dd_or_ds, canonical):
    names = get_label_names(dd_or_ds)
    if names == canonical:
        return dd_or_ds, canonical
    idx = {n: i for i, n in enumerate(canonical)}
    def fix(ex):
        return {"tokens": ex["tokens"],
                "ner_tags": [idx.get(names[i], 0) for i in ex["ner_tags"]]}
    feats = Features({"tokens": Sequence(Value("string")),
                      "ner_tags": Sequence(ClassLabel(names=canonical))})
    if isinstance(dd_or_ds, DatasetDict):
        for s in dd_or_ds:
            dd_or_ds[s] = dd_or_ds[s].map(fix, features=feats)
    else:
        dd_or_ds = dd_or_ds.map(fix, features=feats)
    return dd_or_ds, canonical

canonical_labels = CANDIDATE_LABEL_ORDER
LABEL_LIST = canonical_labels
LABEL2ID   = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL   = {i: l for i, l in enumerate(LABEL_LIST)}
NUM_LABELS = len(LABEL_LIST)


# ============================= CELL 7: Load WikiAnn ===============================
def load_wikiann(lang, canonical):
    dd = robust_load_dataset("unimelb-nlp/wikiann", lang)
    dd, _ = remap_tags_to_canonical(dd, canonical)
    return dd

wikiann_source = load_wikiann(CONFIG["source_lang"], canonical_labels)
if CONFIG["zs_train_cap"] and len(wikiann_source["train"]) > CONFIG["zs_train_cap"]:
    wikiann_source["train"] = (
        wikiann_source["train"].shuffle(seed=42).select(range(CONFIG["zs_train_cap"]))
    )

wikiann_targets = {l: load_wikiann(l, canonical_labels) for l in CONFIG["languages"]}

print("WikiAnn EN train:", len(wikiann_source["train"]))
for l in CONFIG["languages"]:
    print(f"WikiAnn {l} test:", len(wikiann_targets[l]["test"]))


# ============================= CELL 8: Load Naamapadam ============================
def load_naamapadam(lang, canonical, k_shot=None, seed=42, eval_cap=500, val_cap=200):
    dd = robust_load_dataset("ai4bharat/naamapadam", data_dir=lang,
                             revision="refs/convert/parquet", trust_remote_code=False)
    if "validation" not in dd:
        split = dd["train"].train_test_split(test_size=0.1, seed=seed)
        dd["train"], dd["validation"] = split["train"], split["test"]
    dd, _ = remap_tags_to_canonical(dd, canonical)
    if k_shot is not None:
        n = min(k_shot, len(dd["train"]))
        dd["train"] = dd["train"].shuffle(seed=seed).select(range(n))
    if val_cap and len(dd["validation"]) > val_cap:
        dd["validation"] = dd["validation"].shuffle(seed=seed).select(range(val_cap))
    if eval_cap and len(dd["test"]) > eval_cap:
        dd["test"] = dd["test"].shuffle(seed=seed).select(range(eval_cap))
    return dd


# ============================= CELL 9: Training utilities =========================
def make_tokenize_fn(tokenizer):
    def tok_fn(examples):
        tok = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True,
                        max_length=CONFIG["max_len"], padding=False)
        labels_all = []
        for i, labs in enumerate(examples["ner_tags"]):
            word_ids = tok.word_ids(batch_index=i)
            prev, seq = None, []
            for wid in word_ids:
                if wid is None:    seq.append(-100)
                elif wid != prev:  seq.append(labs[wid])
                else:              seq.append(-100)
                prev = wid
            labels_all.append(seq)
        tok["labels"] = labels_all
        return tok
    return tok_fn


def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=2)
    tp, tl = [], []
    for p, l in zip(preds, labels):
        ps = [ID2LABEL[pi] for pi, li in zip(p, l) if li != -100]
        ls = [ID2LABEL[li] for li in l if li != -100]
        tp.append(ps); tl.append(ls)
    res = metric.compute(predictions=tp, references=tl, zero_division=0)
    out = {"precision": res["overall_precision"], "recall": res["overall_recall"],
           "f1": res["overall_f1"], "accuracy": res["overall_accuracy"]}
    for ent in ["PER", "ORG", "LOC"]:
        if ent in res:
            out[f"f1_{ent}"] = res[ent]["f1"]
            out[f"p_{ent}"]  = res[ent]["precision"]
            out[f"r_{ent}"]  = res[ent]["recall"]
        else:
            out[f"f1_{ent}"] = np.nan; out[f"p_{ent}"] = np.nan; out[f"r_{ent}"] = np.nan
    return out


def compute_class_weights(train_ds, num_labels):
    counts = np.ones(num_labels)
    for ex in train_ds:
        for l in ex["ner_tags"]:
            counts[l] += 1
    inv = 1.0 / counts
    return torch.tensor(inv / inv.sum() * num_labels, dtype=torch.float)


class WeightedTrainer(Trainer):
    """Weighted CE to prevent all-O collapse; DataParallel-safe."""
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        m = self.model
        while hasattr(m, "module"): m = m.module
        self._num_labels = int(m.config.num_labels)

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        w = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = CrossEntropyLoss(weight=w, ignore_index=-100)(
            logits.view(-1, self._num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def apply_lalf(model, lang, use_lalf):
    """Language-Adaptive Layer Freezing: freeze bottom-N encoder layers."""
    if not use_lalf:
        return 0
    n = CONFIG["lalf_freeze_by_lang"].get(lang, 0)
    if n <= 0:
        return 0
    enc = None
    for attr in ("bert", "roberta"):
        if hasattr(model, attr):
            base = getattr(model, attr)
            if hasattr(base, "encoder") and hasattr(base.encoder, "layer"):
                enc = base.encoder.layer
                break
    if enc is None:
        return 0
    n = min(n, len(enc))
    for i in range(n):
        for p in enc[i].parameters():
            p.requires_grad = False
    return n


def build_and_train(model_name, ckpt, train_ds, run_name, epochs, batch, seed, lr,
                    use_weights, lang="", init_from_ckpt=None, use_lalf=False):
    set_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(ckpt)
    base = init_from_ckpt if init_from_ckpt else ckpt
    model = AutoModelForTokenClassification.from_pretrained(
        base, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID
    ).to(DEVICE)

    frozen = apply_lalf(model, lang, use_lalf)
    if frozen:
        print(f"   LALF: froze bottom {frozen} layers for lang={lang}")

    tok_fn = make_tokenize_fn(tokenizer)
    train_tok = train_ds.map(tok_fn, batched=True, remove_columns=train_ds.column_names)
    collator = DataCollatorForTokenClassification(tokenizer)

    steps_per_epoch = max(1, len(train_tok) // max(1, batch))
    warmup_steps = max(3, int(CONFIG["warmup_ratio"] * steps_per_epoch * epochs))

    args = TrainingArguments(
        output_dir=os.path.join(CKPT_DIR, run_name),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch,
        per_device_eval_batch_size=32,
        learning_rate=lr,
        weight_decay=CONFIG["weight_decay"],
        warmup_steps=warmup_steps,
        max_grad_norm=CONFIG["max_grad_norm"],
        logging_steps=max(20, steps_per_epoch),
        eval_strategy="no",
        save_strategy="no",
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=seed, data_seed=seed,
        disable_tqdm=True,
    )
    kwargs = dict(model=model, args=args, train_dataset=train_tok,
                  data_collator=collator, compute_metrics=compute_metrics)
    if use_weights:
        kwargs["class_weights"] = compute_class_weights(train_ds, NUM_LABELS)
    trainer = (WeightedTrainer if use_weights else Trainer)(**kwargs)
    trainer.train()
    return trainer, tokenizer


def quick_val_f1(trainer, tokenizer, val_ds):
    tok_fn = make_tokenize_fn(tokenizer)
    vt = val_ds.map(tok_fn, batched=True, remove_columns=val_ds.column_names)
    preds = trainer.predict(vt)
    return compute_metrics((preds.predictions, preds.label_ids))["f1"]


def train_with_guard(model_name, ckpt, train_ds, val_ds, run_name, epochs, batch, seed,
                     lr, use_weights, lang="", init_from_ckpt=None, use_lalf=False):
    """Train, retry once with halved LR if val F1 collapses."""
    attempt, current_lr = 0, lr
    while True:
        trainer, tokenizer = build_and_train(
            model_name, ckpt, train_ds, run_name, epochs, batch, seed,
            current_lr, use_weights, lang=lang,
            init_from_ckpt=init_from_ckpt, use_lalf=use_lalf,
        )
        v_f1 = quick_val_f1(trainer, tokenizer, val_ds)
        collapsed = v_f1 < CONFIG["collapse_f1_threshold"]
        if collapsed and attempt < CONFIG["max_collapse_retries"]:
            attempt += 1
            current_lr *= CONFIG["retry_lr_factor"]
            print(f"   !! val collapse (F1={v_f1:.4f}); retry {attempt} lr={current_lr:.2e}")
            shutil.rmtree(os.path.join(CKPT_DIR, run_name), ignore_errors=True)
            del trainer; gc.collect(); torch.cuda.empty_cache()
            continue
        return trainer, tokenizer, v_f1, attempt


def eval_on(trainer, tokenizer, eval_ds, meta, save_errors=False, tag=""):
    tok_fn = make_tokenize_fn(tokenizer)
    etok = eval_ds.map(tok_fn, batched=True, remove_columns=eval_ds.column_names)
    preds = trainer.predict(etok)
    m = compute_metrics((preds.predictions, preds.label_ids))
    m.update(meta)
    all_results.append(m)
    save_progress()

    collapsed = ((m["f1"] < CONFIG["collapse_f1_threshold"]) and
                 (m["recall"] < CONFIG["collapse_f1_threshold"]))
    flag = "  !! COLLAPSE" if collapsed else ""
    print(f"   -> {meta.get('run')} | eval={meta.get('eval_on')} | "
          f"F1={m['f1']:.4f} (P={m['precision']:.4f} R={m['recall']:.4f}) | "
          f"{time_used_str()}{flag}")
    if collapsed:
        collapsed_runs.append({**meta, "f1": m["f1"]})

    if save_errors and len(error_examples) < 200:
        pid = np.argmax(preds.predictions, axis=2)
        for i, (p_row, l_row) in enumerate(zip(pid, preds.label_ids)):
            ps = [ID2LABEL[p] for p, l in zip(p_row, l_row) if l != -100]
            ls = [ID2LABEL[l] for l in l_row if l != -100]
            if ps != ls:
                error_examples.append({
                    "tag": tag, "tokens": list(eval_ds[i]["tokens"])[:80],
                    "gold": ls[:80], "pred": ps[:80],
                })
                if len(error_examples) >= 200:
                    break


# ============================= CELL 10: PHASE A — Zero-shot =======================
zs_ckpts = {}

if CONFIG["experiments"]["zero_shot"]:
    print("=" * 70)
    print("PHASE A: ZERO-SHOT (EN -> target)")
    print("=" * 70)

    for model_name, ckpt in CONFIG["models"].items():
        for seed in CONFIG["seeds"]:
            run_name = f"zs_{model_name}_s{seed}"
            key = run_key("zero_shot", model_name, "", "", seed)
            ckpt_path = os.path.join(CKPT_DIR, run_name)

            if key in completed_keys and os.path.exists(ckpt_path):
                print(f"\n--- {run_name} : already completed, reusing ckpt ---")
                zs_ckpts[(model_name, seed)] = ckpt_path
                continue

            if time_left() < 900:
                print(f"!! Skipping rest of Phase A ({time_used_str()})")
                break

            print(f"\n--- {run_name} ---")
            try:
                trainer, tok = build_and_train(
                    model_name, ckpt, wikiann_source["train"], run_name,
                    CONFIG["zs_epochs"], CONFIG["zs_batch"], seed, CONFIG["zs_lr"],
                    use_weights=False, use_lalf=False,
                )
                trainer.save_model(ckpt_path); tok.save_pretrained(ckpt_path)
                zs_ckpts[(model_name, seed)] = ckpt_path

                for lang in CONFIG["languages"]:
                    eval_on(
                        trainer, tok, wikiann_targets[lang]["test"],
                        meta=dict(experiment="zero_shot", model=model_name, eval_on=lang,
                                  lang=None, k=None, seed=seed, run=run_name),
                        save_errors=(seed == CONFIG["seeds"][0]),
                        tag=f"zs-{model_name}-{lang}",
                    )
                del trainer; gc.collect(); torch.cuda.empty_cache()
            except Exception as e:
                print(f"   !! {run_name} FAILED entirely: {type(e).__name__}: {str(e)[:300]}")
                failed_runs.append(dict(experiment="zero_shot", model=model_name,
                                        lang="", k="", seed=seed, run=run_name,
                                        error=f"{type(e).__name__}: {str(e)[:300]}"))
                save_failed()
                gc.collect(); torch.cuda.empty_cache()
                continue


# ============================= CELL 11: PHASE B — Few-shot =========================
if CONFIG["experiments"]["few_shot"]:
    print("=" * 70)
    print("PHASE B: FEW-SHOT (target-only, LALF on)")
    print("=" * 70)

    for model_name, ckpt in CONFIG["models"].items():
        for lang in CONFIG["languages"]:
            for k in CONFIG["k_values"]:
                for seed in CONFIG["seeds"]:
                    key = run_key("few_shot", model_name, lang, k, seed)
                    if key in completed_keys:
                        continue
                    if time_left() < CONFIG["min_run_headroom_s"]:
                        print(f"!! Skipping rest of Phase B ({time_used_str()})")
                        break

                    run_name = f"fs_{model_name}_{lang}_k{k}_s{seed}"
                    print(f"\n--- {run_name} ---")
                    try:
                        dd = load_naamapadam(lang, LABEL_LIST, k_shot=k, seed=seed,
                                             eval_cap=CONFIG["test_eval_cap"],
                                             val_cap=CONFIG["val_cap_fewshot"])
                        epochs = CONFIG["fs_epochs_by_k"][k]
                        batch  = min(CONFIG["fs_batch_by_k"][k], max(1, k))
                        lr     = CONFIG["fs_lr_by_model"].get(model_name, 2e-5)

                        trainer, tok, v_f1, retries = train_with_guard(
                            model_name, ckpt, dd["train"], dd["validation"], run_name,
                            epochs, batch, seed, lr, CONFIG["use_weighted_loss"],
                            lang=lang, use_lalf=CONFIG["use_lalf"],
                        )
                        eval_on(
                            trainer, tok, dd["test"],
                            meta=dict(experiment="few_shot", model=model_name, eval_on=lang,
                                      lang=lang, k=k, seed=seed, run=run_name,
                                      retries=retries),
                        )
                        shutil.rmtree(os.path.join(CKPT_DIR, run_name), ignore_errors=True)
                        del trainer; gc.collect(); torch.cuda.empty_cache()
                    except Exception as e:
                        print(f"   !! {run_name} FAILED entirely: {type(e).__name__}: {str(e)[:300]}")
                        failed_runs.append(dict(experiment="few_shot", model=model_name,
                                                lang=lang, k=k, seed=seed, run=run_name,
                                                error=f"{type(e).__name__}: {str(e)[:300]}"))
                        save_failed()
                        shutil.rmtree(os.path.join(CKPT_DIR, run_name), ignore_errors=True)
                        gc.collect(); torch.cuda.empty_cache()
                        continue


# ============================= CELL 12: PHASE C — Sequential =======================
if CONFIG["experiments"]["sequential"]:
    print("=" * 70)
    print("PHASE C: SEQUENTIAL (EN-zero-shot -> target few-shot, LALF on)")
    print("=" * 70)

    for model_name in CONFIG["models"]:
        for seed in CONFIG["seeds"]:
            base = zs_ckpts.get((model_name, seed))
            if base is None:
                print(f"  [skip] no Phase A ckpt for {model_name} seed={seed}")
                continue
            for lang in CONFIG["languages"]:
                for k in CONFIG["k_values"]:
                    key = run_key("sequential", model_name, lang, k, seed)
                    if key in completed_keys:
                        continue
                    if time_left() < CONFIG["min_run_headroom_s"]:
                        print(f"!! Skipping rest of Phase C ({time_used_str()})")
                        break

                    run_name = f"seq_{model_name}_{lang}_k{k}_s{seed}"
                    print(f"\n--- {run_name} ---")
                    try:
                        dd = load_naamapadam(lang, LABEL_LIST, k_shot=k, seed=seed,
                                             eval_cap=CONFIG["test_eval_cap"],
                                             val_cap=CONFIG["val_cap_fewshot"])
                        epochs = CONFIG["fs_epochs_by_k"][k]
                        batch  = min(CONFIG["fs_batch_by_k"][k], max(1, k))
                        lr     = CONFIG["fs_lr_by_model"].get(model_name, 2e-5)

                        trainer, tok, v_f1, retries = train_with_guard(
                            model_name, base, dd["train"], dd["validation"], run_name,
                            epochs, batch, seed, lr, CONFIG["use_weighted_loss"],
                            lang=lang, init_from_ckpt=base, use_lalf=CONFIG["use_lalf"],
                        )
                        eval_on(
                            trainer, tok, dd["test"],
                            meta=dict(experiment="sequential", model=model_name, eval_on=lang,
                                      lang=lang, k=k, seed=seed, run=run_name,
                                      retries=retries),
                        )
                        shutil.rmtree(os.path.join(CKPT_DIR, run_name), ignore_errors=True)
                        del trainer; gc.collect(); torch.cuda.empty_cache()
                    except Exception as e:
                        print(f"   !! {run_name} FAILED entirely: {type(e).__name__}: {str(e)[:300]}")
                        failed_runs.append(dict(experiment="sequential", model=model_name,
                                                lang=lang, k=k, seed=seed, run=run_name,
                                                error=f"{type(e).__name__}: {str(e)[:300]}"))
                        save_failed()
                        shutil.rmtree(os.path.join(CKPT_DIR, run_name), ignore_errors=True)
                        gc.collect(); torch.cuda.empty_cache()
                        continue


# ============================= CELL 13: Aggregate ==================================
results_df = pd.DataFrame(all_results)
results_df.to_csv(RESULTS_CSV, index=False)

if failed_runs:
    print(f"\n!! {len(failed_runs)} runs failed entirely (see failed_runs.csv). "
          f"Re-run this notebook to retry them -- completed runs are skipped automatically.")

if collapsed_runs:
    coll_df = pd.DataFrame(collapsed_runs)
    coll_df.to_csv(os.path.join(OUTPUT_DIR, "collapsed_runs.csv"), index=False)
    print(f"\n!! {len(collapsed_runs)} runs collapsed after retries. See collapsed_runs.csv")
    print(coll_df[["run", "model", "lang", "k", "f1"]].to_string(index=False))
else:
    print("\nNo collapsed runs.")

agg_cols = ["precision", "recall", "f1", "accuracy", "f1_PER", "f1_ORG", "f1_LOC"]
group_keys = ["experiment", "model", "eval_on", "lang", "k"]

mean_df = results_df.groupby(group_keys, dropna=False)[agg_cols].mean().reset_index()
std_df  = results_df.groupby(group_keys, dropna=False)[agg_cols].std().reset_index()
std_df.columns = group_keys + [c + "_std" for c in agg_cols]
summary_df = mean_df.merge(std_df, on=group_keys, how="left")
summary_df.to_csv(os.path.join(OUTPUT_DIR, "summary_mean_std.csv"), index=False)
print("\nSummary (head 20):")
print(summary_df.head(20).to_string())


# ============================= CELL 14: Significance ===============================
def paired_bootstrap(a, b, n=2000, seed=42):
    rng = np.random.default_rng(seed)
    a, b = np.asarray(a), np.asarray(b)
    diffs = np.empty(n)
    for i in range(n):
        idx = rng.integers(0, len(a), len(a))
        diffs[i] = (a[idx] - b[idx]).mean()
    return float((diffs <= 0).mean())

sig_rows = []
for exp in results_df["experiment"].unique():
    sub = results_df[results_df["experiment"] == exp]
    for (lang, k), g in sub.groupby(["lang", "k"], dropna=False):
        pivot = g.pivot_table(index="seed", columns="model", values="f1")
        if len(pivot) < 2:
            continue
        for m1, m2 in itertools.combinations(pivot.columns.tolist(), 2):
            if pivot[m1].notna().all() and pivot[m2].notna().all():
                p = paired_bootstrap(pivot[m1].values, pivot[m2].values)
                # p_value_a_not_gt_b = P(a <= b) under the bootstrap; this is
                # the one-sided p-value for testing "a is not greater than b".
                # LOW value => strong evidence that a IS significantly greater
                # than b. (Older runs called this column "p_a_gt_b", which
                # reads backwards -- fixed here to avoid misreporting in the
                # paper: a low number means "a wins", not "a rarely wins".)
                sig_rows.append({
                    "experiment": exp, "lang": lang, "k": k,
                    "model_a": m1, "model_b": m2,
                    "mean_a": pivot[m1].mean(), "mean_b": pivot[m2].mean(),
                    "p_value_a_not_gt_b": p,
                    "a_significantly_greater_p05": bool(p < 0.05),
                    "n_seeds": len(pivot),
                })
sig_df = pd.DataFrame(sig_rows)
sig_df.to_csv(os.path.join(OUTPUT_DIR, "significance.csv"), index=False)
if not sig_df.empty:
    print(sig_df.to_string())


# ============================= CELL 15: Plots =======================================
def save_bar(pivot, title, fname, ylabel="F1"):
    if pivot.empty: return
    ax = pivot.plot(kind="bar", figsize=(max(7, 1.2 * len(pivot)), 5))
    ax.set_ylabel(ylabel); ax.set_title(title); ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, fname), dpi=150); plt.close()

zs = summary_df[summary_df["experiment"] == "zero_shot"]
if not zs.empty:
    save_bar(zs.pivot_table(index="eval_on", columns="model", values="f1"),
             "Zero-Shot Cross-Lingual NER F1 (EN WikiAnn)", "plot_zeroshot.png")

fs = summary_df[summary_df["experiment"] == "few_shot"]
if not fs.empty:
    for lang in fs["lang"].dropna().unique():
        sub = fs[fs["lang"] == lang].pivot_table(index="k", columns="model", values="f1")
        if not sub.empty:
            ax = sub.plot(marker="o", figsize=(7, 5))
            ax.set_xlabel("k"); ax.set_ylabel("F1")
            ax.set_title(f"Few-shot F1 vs k — {lang.upper()}"); ax.set_ylim(0, 1)
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_DIR, f"plot_fewshot_{lang}.png"), dpi=150)
            plt.close()

seq = summary_df[summary_df["experiment"] == "sequential"]
if not seq.empty:
    for lang in seq["lang"].dropna().unique():
        fs_l, seq_l = fs[fs["lang"] == lang], seq[seq["lang"] == lang]
        if fs_l.empty or seq_l.empty: continue
        fig, ax = plt.subplots(figsize=(8, 5))
        for model in fs_l["model"].unique():
            f_sub = fs_l[fs_l["model"] == model].sort_values("k")
            s_sub = seq_l[seq_l["model"] == model].sort_values("k")
            ax.plot(f_sub["k"], f_sub["f1"], "o--", label=f"{model} (few-shot)")
            ax.plot(s_sub["k"], s_sub["f1"], "s-",  label=f"{model} (seq)")
        ax.set_xlabel("k"); ax.set_ylabel("F1"); ax.set_ylim(0, 1)
        ax.set_title(f"Sequential vs few-shot — {lang.upper()}")
        ax.legend(fontsize=8); plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, f"plot_seq_vs_fs_{lang}.png"), dpi=150)
        plt.close()

if not zs.empty:
    for lang in zs["eval_on"].unique():
        pivot = zs[zs["eval_on"] == lang].set_index("model")[["f1_PER", "f1_ORG", "f1_LOC"]]
        save_bar(pivot, f"Per-entity F1 — zero-shot {lang.upper()}",
                 f"plot_perent_zs_{lang}.png")


# ============================= CELL 16: Error analysis ==============================
if error_examples:
    err_df = pd.DataFrame(error_examples)
    err_df.to_csv(os.path.join(OUTPUT_DIR, "error_examples.csv"), index=False)
    for tag, g in err_df.groupby("tag"):
        print(f"\n=== {tag} (showing 2 of {len(g)}) ===")
        for _, row in g.head(2).iterrows():
            print("TOKENS:", " ".join(row["tokens"][:25]))
            print("GOLD  :", " ".join(row["gold"][:25]))
            print("PRED  :", " ".join(row["pred"][:25]))
            print("-")


# ============================= CELL 17: Package ======================================
print(f"\nTotal run time: {time_used_str()}")
print("Artifacts in:", OUTPUT_DIR)
for f in sorted(os.listdir(OUTPUT_DIR)):
    print("  ", f)

zip_base = "/kaggle/working/NER_Indian_Languages_Q1"
shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print(f"\nZipped: {zip_base}.zip")

Running: /usr/bin/python3 -m pip install -q --upgrade pip setuptools wheel setuptools_scm
Running: /usr/bin/python3 -m pip install -q seqeval evaluate
  OK  numpy==2.0.2
  OK  pandas==2.3.3
  OK  transformers==5.0.0
  OK  datasets==5.0.0
  OK  evaluate==0.4.6
  OK  accelerate==1.13.0
  OK  seqeval (installed)
!! No HF_TOKEN found -- proceeding unauthenticated.
   This worked for languages hi..as last run but is more likely
   to hit transient Hub errors over a long, many-language run.
   Strongly recommended: Kaggle -> Add-ons -> Secrets -> HF_TOKEN
!! FULL RUN -- 12 languages x 4 models x 3 seeds x 3 k-values.
Device: cuda
Models: ['mBERT', 'XLM-R', 'MuRIL', 'IndicBERT']
Langs: ['hi', 'bn', 'mr', 'gu', 'pa', 'or', 'as', 'ur', 'ta', 'te', 'kn', 'ml']
Seeds: [42, 123, 2024] | k: [10, 50, 100]
Experiments: {'zero_shot': True, 'few_shot': True, 'sequential': True}
!! Resume: 936 rows loaded -> 804 unique runs
WikiAnn EN train: 8000
WikiAnn hi test: 1000
WikiAnn bn test: 1000
WikiAnn mr te

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly ini

   LALF: froze bottom 4 layers for lang=hi
{'loss': '1.487', 'grad_norm': '6.898', 'learning_rate': '1.464e-05', 'epoch': '6.667'}
{'loss': '0.5206', 'grad_norm': '9.473', 'learning_rate': '7.5e-06', 'epoch': '13.33'}
{'loss': '0.2743', 'grad_norm': '3.447', 'learning_rate': '3.571e-07', 'epoch': '20'}
{'train_runtime': '4.487', 'train_samples_per_second': '44.58', 'train_steps_per_second': '13.37', 'train_loss': '0.7608', 'epoch': '20'}
   -> fs_mBERT_hi_k10_s42 | eval=hi | F1=0.4314 (P=0.4288 R=0.4341) | 0.01h

--- fs_mBERT_hi_k10_s123 ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly ini

... (1,984 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   LALF: froze bottom 4 layers for lang=te
{'loss': '0.9571', 'grad_norm': '23.44', 'learning_rate': '1.789e-05', 'epoch': '1.538'}
{'loss': '0.4953', 'grad_norm': '7.06', 'learning_rate': '1.368e-05', 'epoch': '3.077'}
{'loss': '0.3011', 'grad_norm': '4.825', 'learning_rate': '9.474e-06', 'epoch': '4.615'}
{'loss': '0.1948', 'grad_norm': '1', 'learning_rate': '5.263e-06', 'epoch': '6.154'}
{'loss': '0.2344', 'grad_norm': '56.29', 'learning_rate': '1.053e-06', 'epoch': '7.692'}
{'train_runtime': '8.209', 'train_samples_per_second': '97.46', 'train_steps_per_second': '12.67', 'train_loss': '0.4255', 'epoch': '8'}
   -> seq_mBERT_te_k100_s2024 | eval=te | F1=0.6280 (P=0.5550 R=0.7232) | 4.15h

--- seq_mBERT_kn_k10_s2024 ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   LALF: froze bottom 4 layers for lang=kn
